In [ ]:
!pip install --upgrade datasets transformers accelerate

In [ ]:
from datasets import load_dataset
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt

# Define a mapping from emotion names to integer IDs
emotion_to_id = {
    'anticipation': 0,
    'sadness': 1,
    'disgust': 2,
    'fear': 3,
    'optimism': 4,
    'anger': 5,
    'joy': 6,
    'surprise': 7
}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    precision = precision_score(labels, preds, average="macro")
    recall = recall_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

def load_myDataset(file_path):
    dataset = load_dataset("csv", data_files=file_path)
    return dataset

def split_dataset(dataset, testSize=0.2):
    train_test_split = dataset["train"].train_test_split(test_size=testSize, seed=42)
    return train_test_split

def set_tokenizer():
    model_name = "distilroberta-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return tokenizer

def tokenize(batch, tokenizer):
    tokens = tokenizer(batch["Reviews"], truncation=True, padding=True)
    # Convert labels from string emotions to integers using the mapping
    tokens["labels"] = [emotion_to_id[label] for label in batch["emotion"]]
    return tokens

def tokenize_dataset(dataset, tokenizer):
    dataset = dataset.map(lambda batch: tokenize(batch, tokenizer), batched=True)
    dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return dataset

def set_model():
    model_name = "distilroberta-base"
    config = AutoConfig.from_pretrained(model_name)

    # Ensure num_labels matches the number of unique emotions
    config.num_labels = len(emotion_to_id) # Should be 8
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
    return model

def set_training_args():
    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=128,
        per_device_eval_batch_size=128,
        num_train_epochs=10,

        weight_decay=0.01,

        # Reverting to the correct parameter 'logging_dir'
        logging_dir="./logs",
        logging_steps=50
        )
    return training_args

def set_trainer(training_args, model, dataset, tokenizer):
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator
        )
    return trainer

def train_test_model(trainer):
    trainer.train()
    trainer.evaluate()
    return trainer

def save_model_tokenizer(trainer, tokenizer):
    trainer.save_model("./my_model")
    tokenizer.save_pretrained("./my_tokenizer")

def draw_metrics(trainer):
    log_history = trainer.state.log_history

    epochs = []
    accuracy = []
    f1 = []
    precision = []
    recall = []

    for log in log_history:
        if "eval_accuracy" in log:
            epochs.append(log["epoch"])
            accuracy.append(log["eval_accuracy"])
            f1.append(log["eval_f1"])
            precision.append(log["eval_precision"])
            recall.append(log["eval_recall"])

    plt.figure(figsize=(12,6))

    plt.plot(epochs, accuracy, marker='o', label="Accuracy")
    plt.plot(epochs, f1, marker='o', label="F1 Score")
    plt.plot(epochs, precision, marker='o', label="Precision")
    plt.plot(epochs, recall, marker='o', label="Recall")

    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Evaluation Metrics per Epoch")
    plt.legend()
    plt.grid(True)
    plt.show()

def main():
    # Set up the components for training and testing the model
    tokenizer = set_tokenizer()
    model = set_model()
    training_args = set_training_args()

    # Prepare the dataset
    dataset = load_myDataset("/content/MovieReviews_Version5.csv")
    dataset = split_dataset(dataset)
    dataset = tokenize_dataset(dataset, tokenizer)

    # Set up the trainer and train/test the model
    trainer = set_trainer(training_args, model, dataset, tokenizer)
    train_test_model(trainer)
    draw_metrics(trainer)

    # Save the trained model and tokenizer
    save_model_tokenizer(trainer, tokenizer)

if __name__ == "__main__":
    main()